# Does the text of an agent skill predict its replication on GitHub?

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/podorize/skillreuse/blob/main/notebooks/reproduce.ipynb)

Companion notebook to the paper of Pedro Odorize and Rafael Allgayer for CMP263 (Machine Learning, UFRGS).
Run it from top to bottom while you read the [paper](https://github.com/podorize/skillreuse/blob/main/paper/latex/main.pdf).
Each part follows one section of the paper: it says what we did and why, shows the code of the study that did it
(the package in `src/skillreuse`), and runs it on the published data. The results printed here are the ones
reported in the section, table or figure named at the top of each part.

Everything from the dated copies to the interpretation of the model runs here. Two steps do not: the download of
GitSkills (7 GB) and the dating of the copies through the GitHub API (about seven hours). Their code is shown, and
their output is read from `data/derived/`. Running all cells takes about 17 minutes on a laptop and about half an hour on Colab; the slowest parts are
5, 6 and 9.

| Part | Paper | Content |
|---|---|---|
| 0 | Appendix A | setup |
| 1 | §3.1–3.2, Figure 1c | the data and how a copy is measured |
| 2 | §3.3 | text attributes |
| 3 | §3.4 | split by author |
| 4 | §3.5, §4.1, Figures 1–3 | exploratory analysis and preprocessing |
| 5 | §3.6, §4.2, Figure 4a | spot-check of seven algorithms |
| 6 | §3.6, §4.2, Table 4, Figure 4b | nested search and model selection |
| 7 | §3.7, §4.2–4.3, Table 5, Figure 5 | final models, threshold and the single evaluation on the holdout |
| 8 | §4.4, Table 6, Appendix A | robustness |
| 9 | §3.8, §4.5, Figures 6–7 | interpretation |
| 10 | §5–6, Appendix A | what may be claimed, and how to reproduce the rest |

---
## Part 0 — Setup
> Paper: Appendix A.

The first cell gets the repository and, on Colab, installs the exact versions listed in `uv.lock`, so that the
results are the same as in the paper. To run the notebook on your own computer, start Jupyter from the project
folder with `uv run jupyter lab`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# the repository: clone it on Colab, or move to its root when the notebook runs from notebooks/
if not Path("src/skillreuse").exists():
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    else:
        if not Path("skillreuse").exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/podorize/skillreuse.git"], check=True)
        os.chdir("skillreuse")
print("working directory:", Path.cwd())

# on Colab, install the versions of uv.lock in this Python (locally, `uv run` already uses them)
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "export", "--frozen", "--no-dev", "--no-hashes", "--no-emit-project",
                    "-o", "requirements.txt"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
import importlib.metadata, inspect, json, time, tomllib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, display
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve

# the versions that produced the paper, next to the ones in use
locked = {p["name"]: p["version"] for p in tomllib.loads(Path("uv.lock").read_text())["package"]}
for name in ("scikit-learn", "numpy", "pandas", "scipy", "shap"):
    print(f"{name:14s} locked {locked[name]:9s} installed {importlib.metadata.version(name)}")
if np.__version__ != importlib.metadata.version("numpy") or pd.__version__ != importlib.metadata.version("pandas"):
    print("An older numpy or pandas was already loaded: restart the session and run all cells again.")

# the code of the study
sys.path.insert(0, "src")
from skillreuse import cohort, collect, features, stats, study

# study.py picks a non-interactive plotting backend for the command line; show the plots here instead
%matplotlib inline
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
DATA = Path("data/derived")


def show_code(function):
    """Show the source code of a function of the study."""
    display(Code(inspect.getsource(function), language="python"))

---
## Part 1 — The data and how a copy is measured
> Paper: [§3.1 and §3.2](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md), Figure 1c.

GitSkills lists every `SKILL.md` found on public GitHub repositories in July 2026. We call each distinct content (one
file hash) a **skill**, and each file with that content in a repository a **copy**. A repository with 100 or more
distinct skills is an **aggregator**: registries and mirrors collect skills in bulk, so a copy there does not show
that a developer chose the skill. The population is every skill with at least one copy outside aggregators, and the
sample is the first 150,000 of them in a fixed pseudo-random order. This is the code that drew the sample. It reads
the full dataset, so it is not run here.

In [ ]:
show_code(collect.build_sample)

Its output is published: every copy of the sampled skills, with the number of distinct skills of its repository.

In [ ]:
occ = pd.read_parquet(DATA / "collection/sample_occurrences.parquet")
population = json.loads(Path("results/phase1/manifest.json").read_text())["counts"]["population_contents"]

print(f"skills outside aggregators (population): {population:,}")
print(f"skills in the sample:                    {occ.content_hash.nunique():,}")
print(f"copies of the sampled skills:            {len(occ):,}")
print(f"  outside aggregators:                   {(~occ.aggregator).sum():,}")
print(f"  in aggregators:                        {occ.aggregator.sum():,}")
occ.head(3)

### Dating a copy

GitSkills says where each copy was, but not when it got there. For every copy outside aggregators we asked the GitHub
API for the history of the file and kept the first commit in which the file has exactly the content of the skill.
`history_query` builds one query for many files at once, and `date_batch` reads up to 500 commits per file:

In [ ]:
show_code(collect.history_query)
show_code(collect.date_batch)

The **arrival** of a copy is that commit date or, if later, the date the repository was created, because a
repository created by cloning another one inherits its old history. A copy that arrived after the end of the
crawl is impossible and counts as not dated:

In [ ]:
show_code(collect.merge_dates)

The dating took about seven hours at the limit of the API, and its result is published. This is what happened to
each copy outside aggregators (Figure 1c). The status codes are stored in Portuguese, so we translate them for the
plot.

In [ ]:
dates = pd.read_parquet(DATA / "collection/dates.parquet")
outside = dates.merge(occ, on=["content_hash", "repo_full_name", "path"]).query("not aggregator")

names = {"datada": "dated", "repositorio_inacessivel": "repository gone", "arquivo_ausente": "file gone",
         "hash_nao_encontrado": "hash not found", "falha_de_coleta": "collection failure",
         "data_posterior_ao_retrato": "date after snapshot"}
share = outside.status.map(names).value_counts(normalize=True)

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.barh(share.index[::-1], share.values[::-1])
for i, value in enumerate(share.values[::-1]):
    ax.text(value + 0.01, i, f"{value:.1%}", va="center", fontsize=8)
ax.set_xlim(0, 1.1)
ax.set_title("Dating of copies outside aggregators (Figure 1c)")
plt.show()
print(f"dated copies: {(outside.status == 'datada').sum():,} of {len(outside):,}")

### First appearance, author and the target

With the dates, each skill gets a **first appearance** (T0, the earliest arrival outside aggregators), an
**origin** (the copy that arrived first) and an **author** (the owner of the origin repository). A skill is
**copied** if a copy by another owner, outside aggregators, arrived within W days of T0. Copies by the author do not
count. `cohort.timing` computes T0, the author and the days until the first copy by another owner:

In [ ]:
show_code(cohort.timing)

The example of Section 3.2: Ana publishes a skill on 2 March, Bruno copies it on 12 March, Ana copies it into a
second project of her own, and a mirror with thousands of skills collects it in April.

In [ ]:
example = pd.DataFrame({
    "content_hash": "skill",
    "repo_full_name": ["ana/project", "bruno/tools", "ana/other-project", "mirror/all-skills"],
    "path": "SKILL.md",
    "owner": ["ana", "bruno", "ana", "mirror"],
    "repo_skills": [1, 3, 1, 5000],
    "arrival": pd.to_datetime(["2026-03-02", "2026-03-12", "2026-03-20", "2026-04-10"], utc=True),
})
example["repo_created_at"] = example["arrival"]

result = cohort.timing(example, limit=100)
display(result[["t0", "author", "first_copy_lag_days"]])
print("copied with W = 60 days:", bool(result.first_copy_lag_days.iloc[0] <= 60))

Ana is the author, Bruno's copy counts (10 days after T0), and her second project and the mirror do not. The same
function on all the dated copies gives T0, the author and the time to first copy of every skill. Each author keeps
at most 50 skills (§3.4), and the result is the table of the study:

In [ ]:
copies = dates.merge(occ[["content_hash", "repo_full_name", "path", "owner", "repo_skills"]],
                     on=["content_hash", "repo_full_name", "path"], how="right")
timing = cohort.timing(copies, limit=100)
table = pd.read_parquet(DATA / "study_table.parquet")

same_t0 = timing.set_index("content_hash").t0.loc[table.content_hash].values == pd.to_datetime(table.t0, utc=True).values
print(f"skills with at least one dated copy:   {len(timing):,}")
print(f"skills after the cap of 50 per author: {len(table):,}")
print("T0 of every skill equals the one in the study table:", bool(same_t0.all()))

---
## Part 2 — Text attributes
> Paper: [§3.3](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md), Table 1.

The model reads only the `SKILL.md`. Since all copies of a skill have the same text, its attributes describe exactly
what other developers saw when they copied it. Each attribute is a count or a ratio computed with a regular
expression, so it can be checked by hand on any file:

In [ ]:
patterns = {name: value.pattern for name, value in vars(features).items() if name.endswith("_PATTERN")}
pd.Series(patterns, name="regular expression").to_frame()

In [ ]:
show_code(features.extract_markdown_features)

The skills' texts are not in the repository (each one keeps the license of its own repository), so here is a small
skill written for the example:

In [ ]:
skill = '''---
name: release-notes
description: Write the release notes of a version from its merged pull requests.
---
# Release notes

Read the merged pull requests since the last tag. Group them by type, and write one line per change.

## Steps
1. List the pull requests with `gh pr list --state merged`.
2. Group them into features, fixes and other changes.
- Keep each line short.

```bash
$ gh pr list --state merged --limit 100
```

See the [style guide](https://example.com/style).
'''
pd.Series(features.extract_markdown_features(skill), name="value").to_frame()

Four more attributes come from the front matter: the length of the name and of the description, and whether each
one is missing. GitSkills gives two more, whether the front matter is valid and the length of the body:

In [ ]:
show_code(features.add_engineered_features)

The 27 text attributes are grouped in six families (Table 1). The families are used again in Part 9, to measure the
importance of a whole group at once:

In [ ]:
pd.Series({family: ", ".join(columns) for family, columns in study.FAMILIES.items()}, name="attributes").to_frame()

A second group describes the **origin repository** and is used only as a control (§3.3, last row of Table 1): its
age at T0, language, license, the folder of the file and its popularity. The stars at T0 could not be rebuilt,
because the GitHub API no longer lists the date of each star, so the control uses the stars and forks of the
repository at the snapshot. The folder attributes come from these patterns:

In [ ]:
print("columns of the control model:", ", ".join(study.ORIGIN_POP))
pd.Series(cohort.PATH_PATTERNS, name="folder pattern").to_frame()

In [ ]:
table[study.TEXT].describe().T[["mean", "50%", "max"]].round(2)

---
## Part 3 — Split by author
> Paper: [§3.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

Skills by the same author look alike, so if an author appeared in both the training and the test data, a model could
simply recognize the author. We split the data by author, before any analysis: 30% of the authors go to the
**holdout** (the test set, used once in Part 7) and the others to the **development set**. The draw is made inside
each size band (1, 2–3, 4–10 and 11–50 skills), in a fixed order given by a hash of the author name:

In [ ]:
show_code(cohort.split_authors)

Running it again on the table of the study gives the same split that the study saved:

In [ ]:
holdout_authors = cohort.split_authors(table.rename(columns={"owner": "author"}))
rebuilt = np.where(table.owner.map(holdout_authors), "holdout", "dev")
print("same split as the study table:", bool((rebuilt == table.split.to_numpy()).all()))

skills_per_author = table.groupby("owner").size()
bands = pd.DataFrame({"band": cohort.size_band(skills_per_author),
                      "holdout": holdout_authors.reindex(skills_per_author.index)})
display(bands.groupby("band", observed=True).holdout.agg(authors="size", share_in_holdout="mean").round(3))

for side in ("dev", "holdout"):
    part = table[table.split == side]
    print(f"{side:8s} {len(part):,} skills from {part.owner.nunique():,} authors")
dev_authors, holdout_authors_set = set(table[table.split == "dev"].owner), set(table[table.split == "holdout"].owner)
print("authors on both sides:", len(dev_authors & holdout_authors_set))
print("most skills of one author:", skills_per_author.max())

---
## Part 4 — Exploratory analysis and preprocessing
> Paper: [§3.5](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figures 1, 2 and 3.

This part uses only the development set, so that nothing from the holdout influences our choices.

### The window W and the target

`load_frame` reads one side of the split and computes the target for a window of W days: a skill is **eligible** if
its whole window ends before the snapshot (9 July 2026), and **copied** (`replicated`) if another developer copied it
within the window.

In [ ]:
show_code(study.load_frame)

W was chosen by a rule written before the analysis: the shortest of 30, 60 and 90 days that covers at least 80% of
the first copies, measured on skills observed for at least 90 days.

In [ ]:
show_code(study.choose_window)

In [ ]:
dev_all = study.load_frame("dev", eligible_only=False)
window_rule = study.choose_window(dev_all)
W = window_rule["W"]
print("share of first copies within W days:", {w: round(share, 3) for w, share in window_rule["coverage"].items()})
print("days to the first copy, quartiles:", window_rule["lag_quantiles"])
print("chosen window W:", W, "days")

In [ ]:
# Figure 1a and 1b
t0 = pd.to_datetime(dev_all.t0, utc=True)
observed = dev_all[t0 <= study.SNAPSHOT - pd.Timedelta(days=90)]
lags = observed.first_copy_lag_days[observed.first_copy_lag_days <= 90]
month = t0.dt.strftime("%Y-%m").where(t0.dt.strftime("%Y-%m") >= "2025-10", "earlier")
per_month = month.value_counts().sort_index(key=lambda m: m.str.replace("earlier", "0"))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
days = np.arange(91)
ax[0].plot(days, [(lags <= d).mean() for d in days])
ax[0].axvline(W, color="red", ls="--", label=f"W = {W} days")
ax[0].axhline(0.8, color="grey", lw=0.7)
ax[0].set(xlabel="days from first appearance to first copy", ylabel="cumulative share",
          title="(a) Time to first copy by another developer")
ax[0].legend()
ax[1].bar(per_month.index, per_month.values)
limit = study.SNAPSHOT - pd.Timedelta(days=W)  # skills that appeared after this date are not eligible
limit_x = list(per_month.index).index(limit.strftime("%Y-%m")) - 0.5 + (limit.day - 1) / limit.days_in_month
ax[1].axvline(limit_x, color="red", ls="--", label=f"eligibility limit ({limit.date()})")
ax[1].tick_params(axis="x", rotation=60, labelsize=7)
ax[1].set_title("(b) First appearance (T0) by month")
ax[1].legend()
plt.show()

In [ ]:
dev = study.load_frame("dev", w=W)
hold = study.load_frame("holdout", w=W)
for name, frame in (("development", dev), ("holdout", hold)):
    print(f"{name:12s} {len(frame):,} eligible skills from {frame.owner.nunique():,} authors, "
          f"{frame.replicated.mean():.1%} copied")

### The attributes

Most counts are zero for many skills and very large for a few (Figure 2a), and many attributes measure almost the same
thing (Figure 2b). The origin attributes have a few missing values; the text attributes have none.

In [ ]:
missing = dev[study.TEXT + study.ORIGIN].isna().mean()
print("columns with missing values:", missing[missing > 0].round(4).to_dict())

In [ ]:
# Figure 2a: skewness before and after log(1 + x); Figure 2b: Spearman correlation
numeric = [c for c in study.TEXT if dev[c].nunique() > 2]
skew_raw = dev[numeric].astype(float).skew()
skew_log = np.log1p(dev[numeric].clip(lower=0).astype(float)).skew()
skewness = pd.DataFrame({"raw": skew_raw, "after log1p": skew_log}).sort_values("raw", ascending=False)
display(skewness.head(8).round(1))

correlation = dev[study.TEXT].astype(float).corr("spearman")
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(study.TEXT)), study.TEXT, rotation=90, fontsize=6)
ax.set_yticks(range(len(study.TEXT)), study.TEXT, fontsize=6)
fig.colorbar(image, ax=ax, shrink=0.7, label="Spearman correlation")
ax.set_title("(2b) Correlation between text attributes")
plt.show()

In [ ]:
# Figure 3: share of copied skills in each quartile of each attribute
rates = {}
for column in [c for c in study.TEXT if dev[c].nunique() > 4]:
    quartile = pd.qcut(dev[column].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
    rates[column] = dev.groupby(quartile, observed=True).replicated.mean()
rates = pd.DataFrame(rates).T
rates.style.format("{:.1%}").background_gradient(cmap="RdBu_r", axis=None)

### From the findings to the preprocessing

Four rules, written before the analysis, turn these findings into preprocessing steps (Table 2): constant columns are
removed; columns with skewness above 2 get a log transform in the models that are sensitive to scale; of two columns
with Spearman correlation above 0.95, only one stays in those models; impossible values become missing.

In [ ]:
show_code(study.eda_rules)

In [ ]:
# a negative age of the repository at T0 is impossible, so it becomes missing
dev_for_rules = dev.assign(repo_age_days_at_t0=dev.repo_age_days_at_t0.where(dev.repo_age_days_at_t0 >= 0))
rules = study.eda_rules(dev_for_rules)
print("constant columns:", rules["constant"])
print("columns with a log transform:", len(rules["skewed_log1p_nontree"]))
print("pairs with Spearman above 0.95:", len(rules["collinear_pairs"]))
print("columns removed for collinearity:", rules["drop_collinear_nontree"])

The rules become a scikit-learn pipeline. Everything is fitted inside each training fold, so no information from the
validation data leaks into training. Trees skip the log, the scaling and the removal of correlated columns.

In [ ]:
show_code(study.preprocessor)
show_code(study.pipeline)

In [ ]:
logistic_text, text_columns = study.pipeline("logistic", "text", rules)
print(len(text_columns), "text columns used by the logistic regression")
logistic_text

---
## Part 5 — Spot-check of seven algorithms
> Paper: [§3.6](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.2](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figure 4a.

Every algorithm is trained and validated on the same partitions of the development set: 3 folds, repeated 3 times
with different seeds, grouped by author and stratified by the target.

In [ ]:
show_code(study.folds)

Seven algorithms, each from a different family, with the default settings of scikit-learn:

In [ ]:
show_code(study.estimator)
show_code(study.clf_metrics)

The loop below trains each algorithm on each training part and measures the AUC on the validation part. It is the
slowest step so far (a few minutes on Colab, most of it in the neural network).

In [ ]:
y_dev = dev.replicated.to_numpy()
rows = []
for repetition, fold, train_idx, valid_idx in study.folds(dev):
    for algo in study.ALGOS:
        model, cols = study.pipeline(algo, "text", rules)
        model.fit(dev.iloc[train_idx][cols], y_dev[train_idx])
        p = model.predict_proba(dev.iloc[valid_idx][cols])[:, 1]
        auc = study.clf_metrics(y_dev[valid_idx], p)["roc_auc"]
        rows.append({"repetition": repetition, "fold": fold, "algorithm": algo, "auc": auc})
spot = pd.DataFrame(rows)

spot_summary = spot.groupby("algorithm").auc.agg(["mean", "std"]).sort_values("mean", ascending=False)
top3 = spot_summary.index[:3].tolist()
display(spot_summary.round(3))
print("the three best go to the nested search:", top3)

In [ ]:
order = spot_summary.index[::-1]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot([spot.auc[spot.algorithm == a] for a in order], orientation="horizontal")
ax.set_yticks(range(1, len(order) + 1), order)
ax.axvline(0.5, color="grey", ls="--", lw=0.8)
ax.set(xlabel="AUC (9 author-grouped folds)", title="(4a) Spot-check, default hyperparameters")
plt.show()

---
## Part 6 — Nested search and model selection
> Paper: [§3.6](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.2](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 4, Figure 4b.

The three best algorithms are tuned with nested cross-validation. In each of six outer folds (the first two
repetitions of the partitions above), a random search tries ten configurations on three inner folds of the training
part and keeps the best. That configuration is then measured on the outer fold, which the search never saw. These are
the search spaces (Table 3) and the search:

In [ ]:
def describe(space):
    # a search space in words: a list of values, or a distribution of scipy
    if isinstance(space, list):
        return "one of " + ", ".join(map(str, space))
    low, high = space.args
    if space.dist.name == "randint":
        return f"integer from {low} to {high - 1}"
    return f"log-uniform from {low:g} to {high:g}"


spaces = [(algo, name, describe(space)) for algo in top3 for name, space in study.SPACES[algo].items()]
display(pd.DataFrame(spaces, columns=["algorithm", "hyperparameter", "values tried"]).set_index(["algorithm", "hyperparameter"]))
show_code(study.inner_cv)
show_code(study.search)

Each algorithm is also measured with its default settings, so six configurations compete. This is the slowest part of
the notebook.

In [ ]:
rows = []
for repetition, fold, train_idx, valid_idx in study.folds(dev, repeats=2):
    train, valid = dev.iloc[train_idx], dev.iloc[valid_idx]
    for algo in top3:
        default_model, cols = study.pipeline(algo, "text", rules)
        start = time.monotonic()
        default_model.fit(train[cols], y_dev[train_idx])
        fit_seconds = time.monotonic() - start
        p = default_model.predict_proba(valid[cols])[:, 1]
        rows.append({"repetition": repetition, "fold": fold, "config": f"{algo}|default",
                     "auc": study.clf_metrics(y_dev[valid_idx], p)["roc_auc"], "fit_seconds": fit_seconds,
                     "train_rows": len(train_idx), "validation_rows": len(valid_idx)})

        tuned = study.search(algo, "text", train, rules, seed=study.SEED + repetition - 1)
        p = tuned.best_estimator_.predict_proba(valid[cols])[:, 1]
        rows.append({"repetition": repetition, "fold": fold, "config": f"{algo}|tuned",
                     "auc": study.clf_metrics(y_dev[valid_idx], p)["roc_auc"], "fit_seconds": tuned.refit_time_,
                     "train_rows": len(train_idx), "validation_rows": len(valid_idx)})
    print("outer fold", repetition, fold, "done")
nested = pd.DataFrame(rows)
nested_summary = nested.groupby("config").agg(mean=("auc", "mean"), std=("auc", "std"),
                                              fit_seconds=("fit_seconds", "mean"))
nested_summary.sort_values("mean", ascending=False).round(3)

Is the best configuration really better than the others? The six outer folds overlap in their training data, so the
paired t test uses the Nadeau–Bengio correction of the variance, and Holm's method adjusts the p-values for the five
comparisons:

In [ ]:
show_code(stats.nb_test)
show_code(stats.holm)
show_code(study.compare_to_best)

In [ ]:
tests = study.compare_to_best(nested.rename(columns={"repetition": "repeat", "auc": "roc_auc"}))
display(tests.round(3))
best = nested_summary["mean"].idxmax()
tied = [best] + tests.loc[tests.p_holm >= 0.05, "other"].tolist()
print("best mean AUC:", best)
print("not significantly different from it:", tied)
print("fastest of the tied configurations on this machine:", nested_summary.loc[tied, "fit_seconds"].idxmin())

The rule of the protocol takes the fastest of the tied configurations, so that extra complexity is accepted only when
it pays off. Fit times depend on the machine, so on Colab the order of the fastest configurations may differ from the
study's. The study recorded its choice before looking at the holdout, and the rest of this notebook uses it:

In [ ]:
winner = json.loads(Path("results/phase4/winner.json").read_text())
print("choice recorded by the study:", winner["winner"])

In [ ]:
# Table 4: spot-check (mean and standard deviation over 9 folds) and nested search (mean over 6 outer folds)
table4 = spot_summary.rename(columns={"mean": "spot-check mean", "std": "spot-check std"})
for variant in ("default", "tuned"):
    table4[f"nested, {variant}"] = [nested_summary["mean"].get(f"{algo}|{variant}") for algo in table4.index]
display(table4.round(3))

order = nested_summary["mean"].sort_values().index
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot([nested.auc[nested.config == c] for c in order], orientation="horizontal")
ax.set_yticks(range(1, len(order) + 1), order)
ax.set(xlabel="AUC (6 outer folds)", title="(4b) Nested tuning of the three finalists")
plt.show()

---
## Part 7 — Final models, threshold and the single evaluation on the holdout
> Paper: [§3.6, §3.7](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md), [§4.2 and §4.3](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 5, Figure 5.

The selected configuration is trained with four sets of columns: the **text** (the model of the research question),
**size only** (does the text say more than its length?), the **origin repository** and **origin repository + text**
(does the text add to where the skill was published?). First, each one is validated on the 9 partitions of the
development set.

In [ ]:
views = ["text", "size", "origin_pop", "origin_pop+text"]
params = {view: study.final_params(view, dev, rules, winner) for view in views}  # empty: default settings
rows, out_of_fold = [], []
for repetition, fold, train_idx, valid_idx in study.folds(dev):
    for view in views:
        model, cols = study.pipeline(winner["algo"], view, rules, params[view])
        model.fit(dev.iloc[train_idx][cols], y_dev[train_idx])
        p = model.predict_proba(dev.iloc[valid_idx][cols])[:, 1]
        rows.append({"view": view, "auc": study.clf_metrics(y_dev[valid_idx], p)["roc_auc"]})
        if view == "text":
            out_of_fold.append(pd.DataFrame({"y": y_dev[valid_idx], "prob": p}))
dev_results = pd.DataFrame(rows)
out_of_fold = pd.concat(out_of_fold)
dev_results.groupby("view", sort=False).auc.agg(["mean", "std"]).round(3)

### The decision threshold

Precision and recall need a threshold on the predicted probability. It is chosen on the development set, before the
holdout is opened: every value from 0.01 to 0.99 is tried on the out-of-fold probabilities of the text model, and
the one with the highest balanced accuracy is kept. Balanced accuracy is the mean of the share of copied skills that
are flagged and the share of other skills that are not, so both classes weigh the same.

In [ ]:
show_code(study.balanced_accuracy)
y_oof, p_oof = out_of_fold.y.to_numpy(), out_of_fold.prob.to_numpy()
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
threshold = float(grid[np.argmax([study.balanced_accuracy(y_oof, p_oof, t) for t in grid])])
at_threshold = study.threshold_report(y_oof, p_oof, threshold)
print("threshold:", threshold)
print({k: round(v, 3) for k, v in at_threshold.items() if k in ("balanced_accuracy", "precision", "recall")})
print(f"share of skills flagged: {(p_oof >= threshold).mean():.1%}")

### The final models

Each model is now trained on the whole development set. The study saved these models, with a fingerprint of each
file, before the holdout was read (`models/` and `results/phase5/freeze.json`). The cell checks that the text model trained
here gives the same predictions as the saved one.

In [ ]:
final_models = {}
for view in views:
    model, cols = study.pipeline(winner["algo"], view, rules, params[view])
    final_models[view] = (model.fit(dev[cols], y_dev), cols)

saved = joblib.load("models/text.joblib")
same = np.allclose(final_models["text"][0].predict_proba(hold[saved["columns"]])[:, 1],
                   saved["pipeline"].predict_proba(hold[saved["columns"]])[:, 1])
print("same predictions as the frozen text model:", same)

### The single evaluation on the holdout

The holdout is used now, once. The 95% intervals come from 2,000 bootstrap samples drawn **by author**, because skills
of the same author are not independent:

In [ ]:
show_code(stats.owner_bootstrap)
show_code(stats.boot_p)

In [ ]:
y_hold, owners = hold.replicated.to_numpy(), hold.owner.astype(str).to_numpy()
probs = {view: model.predict_proba(hold[cols])[:, 1] for view, (model, cols) in final_models.items()}
metrics = pd.DataFrame({view: study.clf_metrics(y_hold, p) for view, p in probs.items()}).T

boot = {view: study.auc_bootstrap(y_hold, p, owners) for view, p in probs.items()}  # about 30 seconds
metrics["auc 95% CI"] = [np.round(study.ci(boot[view]), 3).tolist() for view in metrics.index]
print(f"holdout: {len(hold):,} skills from {hold.owner.nunique():,} authors, {y_hold.mean():.1%} copied")
display(metrics[["roc_auc", "auc 95% CI", "pr_auc", "brier"]].round(4))

The two planned tests: H1, the text model is better than chance (its AUC interval lies above 0.5), and the control,
adding the text to the origin repository raises the AUC (the interval of the gain lies above 0). Holm's method adjusts
the two p-values together.

In [ ]:
gain = boot["origin_pop+text"] - boot["origin_pop"]
over_size = boot["text"] - boot["size"]
p_h1, p_control = stats.holm([stats.boot_p(boot["text"], 0.5), stats.boot_p(gain, 0.0)])
print(f"H1, text model AUC: {metrics.roc_auc['text']:.3f}, 95% CI {np.round(study.ci(boot['text']), 3)}, "
      f"Holm-adjusted p = {p_h1:.3f}")
print(f"gain over size only: {metrics.roc_auc['text'] - metrics.roc_auc['size']:.3f}, "
      f"95% CI {np.round(study.ci(over_size), 3)}")
print(f"control, gain of the text over the origin repository: "
      f"{metrics.roc_auc['origin_pop+text'] - metrics.roc_auc['origin_pop']:.3f}, "
      f"95% CI {np.round(study.ci(gain), 3)}, Holm-adjusted p = {p_control:.3f}")

Figure 5c is the confusion matrix of the text model at the threshold: how many skills it flags, and how many of
those were in fact copied. The precision–recall curves start at a recall of 0.01, because before that every point
rests on a handful of skills.

In [ ]:
at_holdout = study.threshold_report(y_hold, probs["text"], threshold)
flagged = at_holdout["true_positive"] + at_holdout["false_positive"]
print(f"flagged: {flagged:,} | precision {at_holdout['precision']:.3f} | recall {at_holdout['recall']:.3f} "
      f"| balanced accuracy {at_holdout['balanced_accuracy']:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for view in views:
    fpr, tpr, _ = roc_curve(y_hold, probs[view])
    ax[0].plot(fpr, tpr, label=view)
    precision, recall, _ = precision_recall_curve(y_hold, probs[view])
    keep = recall >= 0.01
    ax[1].plot(recall[keep], precision[keep], label=view)
ax[0].plot([0, 1], [0, 1], color="grey", ls="--", lw=0.8)
ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title="(5a) ROC curves")
ax[1].axhline(y_hold.mean(), color="grey", ls="--", lw=0.8, label="share copied")
ax[1].set(xlabel="recall", ylabel="precision", ylim=(0, 1), title="(5b) Precision-recall curves")
for a in ax[:2]:
    a.legend(fontsize=7)
matrix = confusion_matrix(y_hold, (probs["text"] >= threshold).astype(int))
ax[2].imshow(matrix, cmap="Blues")
for (i, j), value in np.ndenumerate(matrix):
    ax[2].text(j, i, f"{value:,}", ha="center", va="center", color="white" if value > matrix.max() / 2 else "black")
ax[2].set_xticks([0, 1], ["predicted\nnot copied", "predicted\ncopied"])
ax[2].set_yticks([0, 1], ["not\ncopied", "copied"])
ax[2].set_title(f"(5c) Text model, threshold {threshold:.2f}")
plt.tight_layout()
plt.show()

---
## Part 8 — Robustness
> Paper: [§4.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Table 6, and [Appendix A](https://github.com/podorize/skillreuse/blob/main/paper/draft/07_appendix_reproducibility.md) for the last two checks.

The answer to RQ1 rests on choices of ours. Each analysis changes one of them and measures the text model again on the
holdout. Changing the window, the definition of aggregator or the period retrains the model on the development set,
with the same configuration:

In [ ]:
show_code(study.retrain_and_evaluate)
show_code(study.auc_with_interval)

In [ ]:
analyses = {}
for other_w in (30, 90):
    analyses[f"window of {other_w} days"] = study.retrain_and_evaluate(
        study.load_frame("dev", other_w), study.load_frame("holdout", other_w), rules)
for variant, label in (("_agg500", "aggregators, limit of 500"), ("_aggnone", "aggregators, no exclusion")):
    analyses[label] = study.retrain_and_evaluate(
        study.load_frame("dev", W, variant), study.load_frame("holdout", W, variant), rules)

# temporal split: learn from the earlier 70% of the development skills, test on holdout skills that came later
cutoff = dev.t0_used.quantile(0.70)
analyses["temporal split"] = study.retrain_and_evaluate(dev[dev.t0_used < cutoff], hold[hold.t0_used >= cutoff], rules)
print("cutoff of the temporal split:", cutoff.date())

# two checks with the final text model, on parts of the holdout
p_text = probs["text"]
dated = hold.all_copies_dated.to_numpy(dtype=bool)
analyses["only skills with every copy dated"] = study.auc_with_interval(y_hold[dated], p_text[dated], owners[dated])
names_in_dev = set(dev.loc[dev.name_norm != "", "name_norm"])
unseen = ~(hold.name_norm.ne("") & hold.name_norm.isin(names_in_dev)).to_numpy()
analyses["only names never seen in development"] = study.auc_with_interval(y_hold[unseen], p_text[unseen], owners[unseen])

robustness = pd.DataFrame(analyses).T[["rows", "prevalence", "auc", "ci95"]]
robustness["ci95"] = robustness.ci95.map(lambda interval: np.round(interval, 3).tolist())
robustness.astype({"auc": float, "prevalence": float}).round({"prevalence": 3, "auc": 3})

---
## Part 9 — What the model learned
> Paper: [§3.8](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) and [§4.5](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md), Figures 6 and 7.

We interpret the text model on 20,000 holdout skills made of whole authors. **Permutation importance** shuffles an
attribute, or a whole family at once, and measures how much the AUC drops: the larger the drop, the more the model
relies on it.

In [ ]:
show_code(study.interp_sample)
show_code(study.permutation_table)

In [ ]:
model, cols = final_models["text"]
sample = study.interp_sample(hold)
y_sample, owners_sample = sample.replicated.to_numpy(), sample.owner.astype(str).to_numpy()
importance = study.permutation_table(model, cols, sample, y_sample, owners_sample, study.SEED, study.N_BOOT_PERM)
print(f"{len(sample):,} skills from {sample.owner.nunique():,} authors")
families = importance[importance.feature.str.startswith("family:")]
attributes = importance[~importance.feature.str.startswith("family:")]
display(families.round(3))
display(attributes.head(10).round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, rows, title in ((ax[0], families, "(6a) Attribute families"), (ax[1], attributes.head(10), "(6b) Top 10 attributes")):
    rows = rows.iloc[::-1]
    a.barh(rows.feature.str.removeprefix("family:"), rows.auc_drop,
           xerr=[rows.auc_drop - rows.ci95_low, rows.ci95_high - rows.auc_drop])
    a.set(xlabel="AUC drop when permuted (95% CI)", title=title)
plt.show()

Is the ranking of the families stable? The same permutation is repeated on the validation part of each of the nine
development folds:

In [ ]:
by_fold = {}
for repetition, fold, train_idx, valid_idx in study.folds(dev):
    fold_model, fold_cols = study.pipeline(winner["algo"], "text", rules)
    fold_model.fit(dev.iloc[train_idx][fold_cols], y_dev[train_idx])
    valid = dev.iloc[valid_idx].reset_index(drop=True)
    drops = study.permutation_table(fold_model, fold_cols, valid, y_dev[valid_idx], valid.owner.astype(str).to_numpy(),
                                    study.SEED + 10 * repetition + fold, n_boot=1, families_only=True)
    by_fold[f"r{repetition}f{fold}"] = drops.set_index("feature").auc_drop
stability = pd.DataFrame(by_fold)
stability["holdout"] = families.set_index("feature").auc_drop
display(stability.rank(ascending=False).astype(int))
spearman = stability.rank().corr("spearman")["holdout"].drop("holdout")
print("Spearman correlation of each fold's ranking with the holdout:", spearman.round(2).tolist())

**SHAP values** split each prediction into the contribution of each attribute, in log-odds. For a logistic
regression, the SHAP value of an attribute is its coefficient times how far the skill is from the average on that
attribute (after the log and the scaling of the pipeline), so it shows the direction and the shape of each effect.

In [ ]:
import shap

x_model = model[:-1].transform(sample[cols])  # the data as the model sees it, after preprocessing
explainer = shap.LinearExplainer(model.named_steps["model"], shap.maskers.Independent(x_model, max_samples=len(x_model)))
shap_values = explainer.shap_values(x_model)
names = list(model.named_steps["pre"].get_feature_names_out())
position = {c: next(k for k, n in enumerate(names) if n.endswith(f"__{c}")) for c in cols}
mean_abs = pd.Series({c: np.abs(shap_values[:, k]).mean() for c, k in position.items()}).sort_values(ascending=False)
display(mean_abs.head(12).round(3).to_frame("mean |SHAP|"))

In [ ]:
# Figure 7a: SHAP values of the 12 most influential attributes, colored by the value of the attribute
top12 = mean_abs.index[:12][::-1]
rng = np.random.default_rng(0)
pick = rng.choice(len(sample), size=5000, replace=False)
fig, ax = plt.subplots(figsize=(10, 5))
for row, c in enumerate(top12):
    values = shap_values[pick, position[c]]
    color = pd.Series(sample[c].to_numpy(dtype=float)[pick]).rank(pct=True)
    ax.scatter(values, row + rng.uniform(-0.3, 0.3, len(values)), c=color, cmap="coolwarm", s=2, alpha=0.5)
ax.set_yticks(range(len(top12)), top12, fontsize=8)
ax.axvline(0, color="grey", lw=0.7)
ax.set(xlabel="SHAP value (log-odds); color from low (blue) to high (red) value of the attribute",
       title="(7a) SHAP values")
plt.show()

# Figure 7b: how the five most important attributes move the prediction
top5 = attributes.feature.head(5).tolist()
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
for a, c in zip(ax, top5):
    x = sample[c].to_numpy(dtype=float)
    low, high = np.nanpercentile(x, [1, 99])
    inside = (x >= low) & (x <= high)
    a.scatter(x[inside], shap_values[inside, position[c]], s=2, alpha=0.3)
    a.axhline(0, color="grey", lw=0.7)
    a.set_title(c, fontsize=9)
ax[0].set_ylabel("SHAP value (log-odds)")
plt.show()

---
## Part 10 — What may be claimed, and how to reproduce the rest
> Paper: [§5 and §6](https://github.com/podorize/skillreuse/blob/main/paper/draft/05_discussion.md), [Appendix A](https://github.com/podorize/skillreuse/blob/main/paper/draft/07_appendix_reproducibility.md).

Section 5.1 states what the study may claim, and each claim rests on an output of this notebook: whether the text
predicts copying at all rests on the test of H1 (Part 7); whether it says more than its length and more than the
origin repository, on the gains over the size-only and the origin models (Part 7); whether this depends on our
choices, on the robustness table (Part 8); and which kinds of attributes matter, on the permutation importance and
its stability across folds (Part 9). Section 5.3 explains what the study cannot claim: it is observational, so it
says nothing about causes, and a model with low precision should not be used to judge single skills.

The repository reproduces the study at four levels, one command each (Appendix A and the README):

| Command | What it redoes | Needs |
|---|---|---|
| `make verify` | applies the frozen models to the holdout and checks the numbers of the paper | the repository |
| `make rerun` | every analysis from the study table on, writing `results/` and `models/` | the repository |
| `make rebuild` | also the sample and the study table, from GitSkills | GitSkills (7 GB) |
| `make from-scratch` | also the dating of the copies through the GitHub API | GitSkills and a GitHub login |